# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
In this module, we're going to get set up with Genie Code. Genie code is your coding assistant with research capabilities. This makes it fast and easy to develop with Databricks using natural language. Genie Code can be used either in global assistant mode or inline to suite your own development style and will help you preview your changes to ensure consistency and reliability.

In this notebook you will:
- Warm up with Genie Code (explain, fix, optimize).
- Create a per-user schema under a shared catalog (e.g., `nova_workshop`). Your schema will be yours to use for this workshop and each user will have their own.


## Part 1 – Genie Code Warm-Up

Use inline Genie (Cmd+I / Ctrl+I) or the side panel.

### Exercise 1 – Explain Code

1. Run the code cell below.
2. Place your cursor in the cell.
3. Press Cmd+I / Ctrl+I and use:

`/explain Explain this code at a beginner level and add inline comments.`

Then accept the changes if you like them.


In [0]:
from pyspark.sql.functions import col, avg

df = spark.read.table("samples.nyctaxi.trips")

agg_df = (
    df.filter(col("trip_distance") > 0)
      .groupBy("pickup_zip")
      .agg(
          avg("trip_distance").alias("avg_trip_distance"),
          avg("fare_amount").alias("avg_fare_amount")
      )
)

display(agg_df)

### Exercise 2 – Debug with Genie

1. Run the cell below (it contains a bug).
2. Let it fail.
3. Click **Diagnose error** in the output or use:

`/fix This code is failing. Explain why and propose a fix.`

Apply the suggested diff if it looks correct.

In [0]:
from pyspark.sql.functions import col

df = spark.read.table("samples.nyctaxi.trips")

# BUG: column name is intentionally wrong
bad_df = df.filter(col("trip_dstance") > 1)

display(bad_df)


### Exercise 3 – Refactor / Optimize

Use:

`/optimize Please optimize this PySpark code for readability and performance. Keep the same logic.`

Review the diff and decide whether to accept.

In [0]:
df = spark.read.table("samples.nyctaxi.trips")

df2 = df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount"
)

from pyspark.sql.functions import col

df3 = df2.withColumn(
    "trip_duration_minutes",
    (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60.0
)

short_trips = df3.filter(
    (col("trip_distance") > 0) &
    (col("trip_distance") < 3) &
    (col("trip_duration_minutes") < 20)
)

display(short_trips)

# Error fixed: column names updated to tpep_pickup_datetime and tpep_dropoff_datetime

## Part 2 – Per-User Schema Setup

We will use a shared catalog, e.g. `nova_workshop`, with one schema per user.

1. Decide a schema name, e.g. `andrij_demo`.
2. Set it below and run the cell.
3. Optional: Use `/explain` to have Genie describe the SQL.

In [0]:
# Set your personal schema name here
user_schema = "andrij_demo"  # TODO: edit this per user
catalog_name = "nova_workshop"

print(f"Using catalog = {catalog_name}, schema = {user_schema}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{user_schema}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {user_schema}")